# Error analysis — verify UNI-65 artifacts

Four things to check:
1. The three prompt YAMLs that feed `infer_relations.py`.
2. The three per-condition JSONLs from Snellius (`data/intermediate/llama_runs/`) — schema, source, parse_error, response_raw, relation counts.
3. The composer output (`temporal_causal_independent.jsonl`) — `source=composed`, relations are union of temporal + causal.
4. The merged `data/results/experiment.jsonl` — all four condition slots per summary.


In [1]:
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROMPTS_DIR = ROOT / "models" / "llama" / "prompts"
LLAMA_RUNS  = ROOT / "data" / "intermediate" / "llama_runs"
EXPERIMENT  = ROOT / "data" / "results" / "experiment.jsonl"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)
print(f"ROOT = {ROOT}")

ROOT = /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis


## 1. The three prompt YAMLs

In [2]:
for cond in ("temporal", "causal", "temporal_causal_joint"):
    path = PROMPTS_DIR / f"{cond}.yaml"
    print(f"=== {path} ===")
    print(path.read_text())
    print()

=== /Users/bugrasipahioglu/repo/uva-thesis-computational-narrative-analysis/models/llama/prompts/temporal.yaml ===
condition: temporal

system: |
  You are an expert computational narratologist. Given a short narrative whose
  events have been pre-extracted and inlined as [eID|trigger|EVENT_TYPE] markers,
  identify the TEMPORAL relations between these events (if such relation(s) exists).

  CODEBOOK: TEMPORAL TAXONOMY
  BEFORE:   The source event completely finishes before the target event starts.
  OVERLAPS: The source event and target event happen at the same time or intersect.
  CONTAINS: The source event completely encapsulates the target event in time.
  IDENTITY: The source event and target event refer to the exact same occurrence.

  FORMATTING CONSTRAINTS
  Output STRICTLY a valid JSON object with this shape:
    {"temporal_relations": [
       {"source": "eID", "target": "eID", "relation": "RELATION_NAME"}
     ]}
  Do not include explanations or conversational text.
  Refer 

## 2. Per-condition JSONLs from the cluster

Confirms every condition_block field is present (schema), parse errors are recorded, response_raw is kept verbatim, and per-row relation counts.

In [3]:
for cond in ("temporal", "causal", "temporal_causal_joint"):
    path = LLAMA_RUNS / f"{cond}.jsonl"
    rows = [json.loads(line) for line in path.read_text().splitlines() if line]
    print(f"=== {cond} ({len(rows)} rows) ===")
    cb_keys = sorted(rows[0]["condition_block"].keys())
    print(f"  condition_block keys: {cb_keys}")
    for i, r in enumerate(rows):
        cb = r["condition_block"]
        rel = cb["relations"] or {}
        rel_counts = {k: len(v) for k, v in rel.items()}
        print(
            f"  row {i}: wid={r['wikidata_id']:<10} sid={r['summary_id']:<5} "
            f"src={cb['source']:<10} "
            f"in_tok={cb['input_tokens']!s:>5} out_tok={cb['output_tokens']!s:>5} "
            f"max_new={cb['max_new_tokens']!s:>5} cap={cb['hit_ctx_cap']!s:<5} "
            f"resp_len={len(cb['response_raw'] or '')!s:>6} "
            f"parse_err={cb['parse_error']!r} "
            f"rel={rel_counts}"
        )
    print()

=== temporal (10 rows) ===
  condition_block keys: ['hit_ctx_cap', 'input_tokens', 'max_new_tokens', 'model_id', 'output_tokens', 'parse_error', 'prompt_rendered', 'prompt_template', 'relations', 'response_parsed', 'response_raw', 'source']
  row 0: wid=1000352    sid=en    src=llama      in_tok= 2938 out_tok= 3331 max_new= 5254 cap=False resp_len=  9184 parse_err=None rel={'temporal_relations': 151}
  row 1: wid=1000352    sid=de    src=llama      in_tok= 1504 out_tok= 1175 max_new= 6688 cap=False resp_len=  3155 parse_err=None rel={'temporal_relations': 1}
  row 2: wid=1000826    sid=en    src=llama      in_tok= 1088 out_tok=  339 max_new= 7104 cap=False resp_len=   925 parse_err=None rel={'temporal_relations': 15}
  row 3: wid=1000826    sid=de    src=llama      in_tok=  736 out_tok=  516 max_new= 7456 cap=False resp_len=  1399 parse_err=None rel={'temporal_relations': 23}
  row 4: wid=1000826    sid=it    src=llama      in_tok= 1031 out_tok=  647 max_new= 7161 cap=False resp_len=  

## 3. Composer output — temporal + causal → temporal_causal_independent

Each composed row should have `source=composed`, `composed_from=['temporal','causal']`, and `relations` = union of the temporal_relations and causal_relations from the upstream runs.

In [4]:
path = LLAMA_RUNS / "temporal_causal_independent.jsonl"
rows = [json.loads(line) for line in path.read_text().splitlines() if line]
print(f"=== composed tcindep ({len(rows)} rows) ===")

# Load temporal + causal once for cross-check.
by_key = {}
for cond in ("temporal", "causal"):
    for r in (json.loads(l) for l in (LLAMA_RUNS / f"{cond}.jsonl").read_text().splitlines() if l):
        by_key[(cond, r["wikidata_id"], r["summary_id"])] = r

for i, r in enumerate(rows):
    cb = r["condition_block"]
    rel = cb["relations"]
    n_t = len(rel.get("temporal_relations", []))
    n_c = len(rel.get("causal_relations", []))
    # Cross-check against upstream.
    t_up = by_key.get(("temporal", r["wikidata_id"], r["summary_id"]))
    c_up = by_key.get(("causal",   r["wikidata_id"], r["summary_id"]))
    n_t_up = len((t_up["condition_block"]["relations"] or {}).get("temporal_relations", []))
    n_c_up = len((c_up["condition_block"]["relations"] or {}).get("causal_relations",   []))
    match = "OK" if (n_t == n_t_up and n_c == n_c_up) else "MISMATCH"
    print(
        f"  row {i}: wid={r['wikidata_id']:<10} sid={r['summary_id']:<5} "
        f"source={cb['source']:<10} composed_from={cb.get('composed_from')} "
        f"temporal_relations={n_t} (upstream={n_t_up})  causal_relations={n_c} (upstream={n_c_up})  [{match}]"
    )

=== composed tcindep (10 rows) ===
  row 0: wid=1000352    sid=de    source=composed   composed_from=['temporal', 'causal'] temporal_relations=1 (upstream=1)  causal_relations=8 (upstream=8)  [OK]
  row 1: wid=1000352    sid=en    source=composed   composed_from=['temporal', 'causal'] temporal_relations=151 (upstream=151)  causal_relations=136 (upstream=136)  [OK]
  row 2: wid=1000826    sid=de    source=composed   composed_from=['temporal', 'causal'] temporal_relations=23 (upstream=23)  causal_relations=23 (upstream=23)  [OK]
  row 3: wid=1000826    sid=en    source=composed   composed_from=['temporal', 'causal'] temporal_relations=15 (upstream=15)  causal_relations=8 (upstream=8)  [OK]
  row 4: wid=1000826    sid=it    source=composed   composed_from=['temporal', 'causal'] temporal_relations=29 (upstream=29)  causal_relations=23 (upstream=23)  [OK]
  row 5: wid=1001400    sid=en    source=composed   composed_from=['temporal', 'causal'] temporal_relations=40 (upstream=40)  causal_rela

## 4. Merged `experiment.jsonl` — one row per summary, all four conditions nested

One DataFrame row per `(wikidata_id, summary_id, condition)`. Embeddings / baselines are reserved (`{}`) until the downstream stages populate them.

In [5]:
exp_rows = [json.loads(line) for line in EXPERIMENT.read_text().splitlines() if line]
print(f"experiment.jsonl: {len(exp_rows)} rows  (embeddings/baselines reserved as empty dicts)\n")

flat = []
for r in exp_rows:
    for cond, cb in r["conditions"].items():
        rel = cb.get("relations") or {}
        flat.append({
            "wikidata_id":    r["wikidata_id"],
            "summary_id":     r["summary_id"],
            "n_events":       len(r["events"]),
            "condition":      cond,
            "source":         cb["source"],
            "input_tokens":   cb.get("input_tokens"),
            "output_tokens":  cb.get("output_tokens"),
            "hit_ctx_cap":    cb.get("hit_ctx_cap"),
            "parse_error":    cb.get("parse_error"),
            "n_relations":    sum(len(v) for v in rel.values()),
            "prompt_len":     len(cb.get("prompt_rendered") or ""),
            "response_len":   len(cb.get("response_raw") or ""),
        })
pd.DataFrame(flat)

experiment.jsonl: 10 rows  (embeddings/baselines reserved as empty dicts)



,wikidata_id,summary_id,n_events,condition,source,input_tokens,output_tokens,hit_ctx_cap,parse_error,n_relations,prompt_len,response_len
0,1000352,de,62,temporal,llama,1504.0,1175.0,False,NaN,1,5423,3155
1,1000352,de,62,causal,llama,1500.0,185.0,False,NaN,8,5403,498
2,1000352,de,62,temporal_causal_joint,llama,1718.0,336.0,False,NaN,14,6438,981
3,1000352,de,62,temporal_causal_independent,composed,NaN,NaN,None,NaN,9,0,0
4,1000352,en,155,temporal,llama,2938.0,3331.0,False,NaN,151,9862,9184
5,1000352,en,155,causal,llama,2934.0,3819.0,False,NaN,136,9842,11697
6,1000352,en,155,temporal_causal_joint,llama,3152.0,1812.0,False,JSONDecodeError: Expecting value: line 1 column 1 (char 0),0,10877,5251
7,1000352,en,155,temporal_causal_independent,composed,NaN,NaN,None,NaN,287,0,0
8,1000826,de,26,temporal,llama,736.0,516.0,False,NaN,23,2798,1399
9,1000826,de,26,causal,llama,732.0,516.0,False,NaN,23,2778,1378


## 5. Full content of one experiment.jsonl row

The DataFrame above hides the large fields (text, sentences, events, prompt_rendered, response_raw, relations). The cell below dumps **all of row 0** with long strings truncated for display so you can visually confirm every payload is present in the file. Change `pick_idx` to inspect a different summary.

In [6]:
pick_idx = 0
row = exp_rows[pick_idx]

# Helper: copy the row but truncate long string fields so the JSON dump is readable.
def _shorten(obj, limit=300):
    if isinstance(obj, str):
        return obj if len(obj) <= limit else obj[:limit] + f"... ({len(obj) - limit} more chars)"
    if isinstance(obj, list):
        return [_shorten(x, limit) for x in obj]
    if isinstance(obj, dict):
        return {k: _shorten(v, limit) for k, v in obj.items()}
    return obj

print(f"=== experiment.jsonl row {pick_idx}: wid={row['wikidata_id']} sid={row['summary_id']} ===\n")
print(f"top-level keys: {sorted(row.keys())}")
print(f"conditions    : {sorted(row['conditions'].keys())}")
print(f"n events      : {len(row['events'])}")
print(f"n sentences   : {len(row['sentences'])}")
print(f"embeddings    : {row['embeddings']}")
print(f"baselines     : {row['baselines']}")
print()
print("--- Full row (long strings truncated to 300 chars for display; the file on disk has them in full) ---\n")
print(json.dumps(_shorten(row), indent=2, ensure_ascii=False))

=== experiment.jsonl row 0: wid=1000352 sid=de ===

top-level keys: ['baselines', 'conditions', 'embeddings', 'events', 'genres', 'lang', 'linearized_events_only', 'n_sentences', 'n_tokens', 'sentences', 'split', 'summary_id', 'text', 'wikidata_id']
conditions    : ['causal', 'temporal', 'temporal_causal_independent', 'temporal_causal_joint']
n events      : 62
n sentences   : 28
embeddings    : {}
baselines     : {}

--- Full row (long strings truncated to 300 chars for display; the file on disk has them in full) ---

{
  "wikidata_id": "1000352",
  "summary_id": "de",
  "lang": "de",
  "text": "New York: As a 12-year-old boy, Jacob, longing for his missing father, John Reckless, enters his room. There, he finds a message that takes him through a mirror into a world of mirrors. Where the characters of Grimm's Fairy Tales actually exist, but the world has evolved and modernized and is now in... (2992 more chars)",
  "sentences": [
    "New York: As a 12-year-old boy, Jacob, longing for

## 6. Failure inspection — did we capture every hallucination / parse error?

Two distinct failure buckets after UNI-65:

- **A. JSON-decode failures** (`condition_block.parse_error` populated). Llama produced text that doesn't parse as JSON — e.g. prose-wrapped output or mid-string truncation. The verbatim model output sits in `response_raw`.
- **B. Silent drops** (per UNI-65 Option A). JSON parsed fine, but some relations had unknown labels or malformed eIDs and were dropped from `relations`. The full set is still in `response_raw` — this cell diffs raw vs kept to show exactly what got dropped.

In [7]:
print("=== A. parse_error rows (JSON-decode failures) ===\n")
n_parse_errors = 0
for r in exp_rows:
    for cond, cb in r["conditions"].items():
        if not cb.get("parse_error"):
            continue
        n_parse_errors += 1
        raw = cb.get("response_raw") or ""
        print(f"--- wid={r['wikidata_id']} sid={r['summary_id']} cond={cond} ---")
        print(f"parse_error: {cb['parse_error']}")
        print(f"response_raw ({len(raw)} chars, first 2000 shown):")
        print(raw[:2000] + (f"\n... ({len(raw) - 2000} more chars)" if len(raw) > 2000 else ""))
        print()
if n_parse_errors == 0:
    print("  (none)\n")

print("\n=== B. Silently-dropped relations (bad labels / bad eIDs) ===\n")
n_drops = 0
for r in exp_rows:
    for cond, cb in r["conditions"].items():
        if cb["source"] != "llama" or not cb.get("response_raw") or cb.get("parse_error"):
            continue
        try:
            raw_parsed = json.loads(cb["response_raw"])
        except json.JSONDecodeError:
            continue
        kept = cb.get("relations") or {}
        kept_set = {(rel["source"], rel["target"], rel["relation"])
                    for v in kept.values() for rel in v}
        dropped_here = []
        for section, rels in raw_parsed.items():
            if not isinstance(rels, list):
                continue
            for rel in rels:
                if not isinstance(rel, dict):
                    continue
                key = (rel.get("source"), rel.get("target"), rel.get("relation"))
                if key not in kept_set:
                    dropped_here.append((section, rel))
        if not dropped_here:
            continue
        n_drops += 1
        kept_count = sum(len(v) for v in kept.values())
        raw_count = sum(len(v) for v in raw_parsed.values() if isinstance(v, list))
        print(f"--- wid={r['wikidata_id']} sid={r['summary_id']} cond={cond} ---")
        print(f"raw rels in response: {raw_count}, kept: {kept_count}, dropped: {len(dropped_here)}")
        for section, rel in dropped_here:
            print(f"  DROPPED [{section}]: {rel}")
        print()
if n_drops == 0:
    print("  (none)\n")

=== A. parse_error rows (JSON-decode failures) ===

--- wid=1000352 sid=en cond=temporal_causal_joint ---
parse_error: JSONDecodeError: Expecting value: line 1 column 1 (char 0)
response_raw (5251 chars, first 2000 shown):
Here is the JSON output:

{
"joint_relations": [
  {"source": "e2", "target": "e3", "relation": "CAUSE_BEFORE"},
  {"source": "e4", "target": "e5", "relation": "CAUSE_OVERLAPS"},
  {"source": "e5", "target": "e6", "relation": "CAUSE_OVERLAPS"},
  {"source": "e7", "target": "e8", "relation": "CAUSE_BEFORE"},
  {"source": "e9", "target": "e10", "relation": "CAUSE_BEFORE"},
  {"source": "e11", "target": "e12", "relation": "CAUSE_BEFORE"},
  {"source": "e13", "target": "e14", "relation": "CAUSE_BEFORE"},
  {"source": "e15", "target": "e16", "relation": "CAUSE_BEFORE"},
  {"source": "e17", "target": "e18", "relation": "CAUSE_BEFORE"},
  {"source": "e19", "target": "e20", "relation": "CAUSE_BEFORE"},
  {"source": "e21", "target": "e22", "relation": "CAUSE_BEFORE"},
  {"sou

## 7. UNI-26: Linearization spot-check

Spot-checks the linearizations on the current `experiment.jsonl` sample.
This is human eyeballing, not metrics — those live in UNI-71 / UNI-28.


In [8]:
import json
from pathlib import Path
import pandas as pd

EXP_PATH = Path("../data/results/experiment.jsonl")
df = pd.read_json(EXP_PATH, lines=True)
print(f"rows: {len(df)}")
df[["wikidata_id", "summary_id", "lang", "split"]]


rows: 10


,wikidata_id,summary_id,lang,split
0,1000352,de,de,train
1,1000352,en,en,train
2,1000826,de,de,train
3,1000826,en,en,train
4,1000826,it,it,train
5,1001400,en,en,train
6,1001400,fr,fr,train
7,100156260,en,en,train
8,100156260,fr,fr,train
9,100156260,it,it,train


In [9]:
from IPython.display import Markdown, display

PREVIEW = 400

def _truncate(s: str, n: int = PREVIEW) -> str:
    if len(s) <= n:
        return s
    return s[:n] + f"\n…(truncated, full length {len(s)} chars)"

for _, row in df.iterrows():
    nrel = {
        "temporal":   len((row["conditions"]["temporal"]                   ["relations"] or {}).get("temporal_relations", [])),
        "causal":     len((row["conditions"]["causal"]                     ["relations"] or {}).get("causal_relations",   [])),
        "joint":      len((row["conditions"]["temporal_causal_joint"]      ["relations"] or {}).get("joint_relations",    [])),
        "indep(t)":   len((row["conditions"]["temporal_causal_independent"]["relations"] or {}).get("temporal_relations", [])),
        "indep(c)":   len((row["conditions"]["temporal_causal_independent"]["relations"] or {}).get("causal_relations",   [])),
    }
    md = [f"### {row['wikidata_id']} / {row['summary_id']} ({row['lang']}) — events: {len(row['events'])}, relations: {nrel}"]
    md.append("**events_only**\n```\n" + _truncate(row["linearized_events_only"]) + "\n```")
    for cond in ("temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"):
        md.append(f"**{cond}**\n```\n" + _truncate(row["conditions"][cond]["linearized"]) + "\n```")
    display(Markdown("\n\n".join(md)))


### 1000352 / de (de) — events: 62, relations: {'temporal': 1, 'causal': 8, 'joint': 14, 'indep(t)': 1, 'indep(c)': 8}

**events_only**
```
EVENTS:
(e1|enters|Arriving) (e2|finds|Know) (e3|exist|Presence) (e4|evolved|Coming_to_be) (e5|modernized|Cause_to_make_progress) (e6|visits|Traveling) (e7|sneak|Self_motion) (e8|becomes|Becoming) (e9|becomes|Becoming) (e10|entered|Arriving) (e11|attacked|Attack) (e12|causes|Causation) (e13|transforms|Change) (e14|sets off|Departing) (e15|find|Know) (e16|escape|Escaping) (e17|makes|Manufacturing) 
…(truncated, full length 1422 chars)
```

**temporal**
```
EVENTS:
(e1|enters|Arriving) (e2|finds|Know) (e3|exist|Presence) (e4|evolved|Coming_to_be) (e5|modernized|Cause_to_make_progress) (e6|visits|Traveling) (e7|sneak|Self_motion) (e8|becomes|Becoming) (e9|becomes|Becoming) (e10|entered|Arriving) (e11|attacked|Attack) (e12|causes|Causation) (e13|transforms|Change) (e14|sets off|Departing) (e15|find|Know) (e16|escape|Escaping) (e17|makes|Manufacturing) 
…(truncated, full length 1450 chars)
```

**causal**
```
EVENTS:
(e1|enters|Arriving) (e2|finds|Know) (e3|exist|Presence) (e4|evolved|Coming_to_be) (e5|modernized|Cause_to_make_progress) (e6|visits|Traveling) (e7|sneak|Self_motion) (e8|becomes|Becoming) (e9|becomes|Becoming) (e10|entered|Arriving) (e11|attacked|Attack) (e12|causes|Causation) (e13|transforms|Change) (e14|sets off|Departing) (e15|find|Know) (e16|escape|Escaping) (e17|makes|Manufacturing) 
…(truncated, full length 1574 chars)
```

**temporal_causal_independent**
```
EVENTS:
(e1|enters|Arriving) (e2|finds|Know) (e3|exist|Presence) (e4|evolved|Coming_to_be) (e5|modernized|Cause_to_make_progress) (e6|visits|Traveling) (e7|sneak|Self_motion) (e8|becomes|Becoming) (e9|becomes|Becoming) (e10|entered|Arriving) (e11|attacked|Attack) (e12|causes|Causation) (e13|transforms|Change) (e14|sets off|Departing) (e15|find|Know) (e16|escape|Escaping) (e17|makes|Manufacturing) 
…(truncated, full length 1602 chars)
```

**temporal_causal_joint**
```
EVENTS:
(e1|enters|Arriving) (e2|finds|Know) (e3|exist|Presence) (e4|evolved|Coming_to_be) (e5|modernized|Cause_to_make_progress) (e6|visits|Traveling) (e7|sneak|Self_motion) (e8|becomes|Becoming) (e9|becomes|Becoming) (e10|entered|Arriving) (e11|attacked|Attack) (e12|causes|Causation) (e13|transforms|Change) (e14|sets off|Departing) (e15|find|Know) (e16|escape|Escaping) (e17|makes|Manufacturing) 
…(truncated, full length 1819 chars)
```

### 1000352 / en (en) — events: 155, relations: {'temporal': 151, 'causal': 136, 'joint': 0, 'indep(t)': 151, 'indep(c)': 136}

**events_only**
```
EVENTS:
(e1|opens|Openness) (e2|using|Using) (e3|enter|Arriving) (e4|attacked|Attack) (e5|begins|Process_start) (e6|turning|Becoming) (e7|knows|Know) (e8|invade|Attack) (e9|become|Becoming) (e10|rides|Self_motion) (e11|told|Telling) (e12|sneak|Motion) (e13|leading to|Causation) (e14|going|Self_motion) (e15|to|Motion) (e16|entering|Arriving) (e17|journey|Self_motion) (e18|deserted|Escaping) (e19|ar
…(truncated, full length 3641 chars)
```

**temporal**
```
EVENTS:
(e1|opens|Openness) (e2|using|Using) (e3|enter|Arriving) (e4|attacked|Attack) (e5|begins|Process_start) (e6|turning|Becoming) (e7|knows|Know) (e8|invade|Attack) (e9|become|Becoming) (e10|rides|Self_motion) (e11|told|Telling) (e12|sneak|Motion) (e13|leading to|Causation) (e14|going|Self_motion) (e15|to|Motion) (e16|entering|Arriving) (e17|journey|Self_motion) (e18|deserted|Escaping) (e19|ar
…(truncated, full length 6616 chars)
```

**causal**
```
EVENTS:
(e1|opens|Openness) (e2|using|Using) (e3|enter|Arriving) (e4|attacked|Attack) (e5|begins|Process_start) (e6|turning|Becoming) (e7|knows|Know) (e8|invade|Attack) (e9|become|Becoming) (e10|rides|Self_motion) (e11|told|Telling) (e12|sneak|Motion) (e13|leading to|Causation) (e14|going|Self_motion) (e15|to|Motion) (e16|entering|Arriving) (e17|journey|Self_motion) (e18|deserted|Escaping) (e19|ar
…(truncated, full length 6204 chars)
```

**temporal_causal_independent**
```
EVENTS:
(e1|opens|Openness) (e2|using|Using) (e3|enter|Arriving) (e4|attacked|Attack) (e5|begins|Process_start) (e6|turning|Becoming) (e7|knows|Know) (e8|invade|Attack) (e9|become|Becoming) (e10|rides|Self_motion) (e11|told|Telling) (e12|sneak|Motion) (e13|leading to|Causation) (e14|going|Self_motion) (e15|to|Motion) (e16|entering|Arriving) (e17|journey|Self_motion) (e18|deserted|Escaping) (e19|ar
…(truncated, full length 9179 chars)
```

**temporal_causal_joint**
```
EVENTS:
(e1|opens|Openness) (e2|using|Using) (e3|enter|Arriving) (e4|attacked|Attack) (e5|begins|Process_start) (e6|turning|Becoming) (e7|knows|Know) (e8|invade|Attack) (e9|become|Becoming) (e10|rides|Self_motion) (e11|told|Telling) (e12|sneak|Motion) (e13|leading to|Causation) (e14|going|Self_motion) (e15|to|Motion) (e16|entering|Arriving) (e17|journey|Self_motion) (e18|deserted|Escaping) (e19|ar
…(truncated, full length 3656 chars)
```

### 1000826 / de (de) — events: 26, relations: {'temporal': 23, 'causal': 23, 'joint': 23, 'indep(t)': 23, 'indep(c)': 23}

**events_only**
```
EVENTS:
(e1|ruled|Control) (e2|arrested|Arrest) (e3|escape|Escaping) (e4|negotiate|Communication) (e5|hire|Employment) (e6|re|Cause_to_amalgamate) (e7|leaving|Departing) (e8|named|Name_conferral) (e9|joins|Becoming_a_member) (e10|realizes|Know) (e11|raids|Attack) (e12|recruiting|Becoming_a_member) (e13|support|Supporting) (e14|attack|Attack) (e15|imprisoned|Arrest) (e16|take|Conquering) (e17|out|A
…(truncated, full length 634 chars)
```

**temporal**
```
EVENTS:
(e1|ruled|Control) (e2|arrested|Arrest) (e3|escape|Escaping) (e4|negotiate|Communication) (e5|hire|Employment) (e6|re|Cause_to_amalgamate) (e7|leaving|Departing) (e8|named|Name_conferral) (e9|joins|Becoming_a_member) (e10|realizes|Know) (e11|raids|Attack) (e12|recruiting|Becoming_a_member) (e13|support|Supporting) (e14|attack|Attack) (e15|imprisoned|Arrest) (e16|take|Conquering) (e17|out|A
…(truncated, full length 1072 chars)
```

**causal**
```
EVENTS:
(e1|ruled|Control) (e2|arrested|Arrest) (e3|escape|Escaping) (e4|negotiate|Communication) (e5|hire|Employment) (e6|re|Cause_to_amalgamate) (e7|leaving|Departing) (e8|named|Name_conferral) (e9|joins|Becoming_a_member) (e10|realizes|Know) (e11|raids|Attack) (e12|recruiting|Becoming_a_member) (e13|support|Supporting) (e14|attack|Attack) (e15|imprisoned|Arrest) (e16|take|Conquering) (e17|out|A
…(truncated, full length 1051 chars)
```

**temporal_causal_independent**
```
EVENTS:
(e1|ruled|Control) (e2|arrested|Arrest) (e3|escape|Escaping) (e4|negotiate|Communication) (e5|hire|Employment) (e6|re|Cause_to_amalgamate) (e7|leaving|Departing) (e8|named|Name_conferral) (e9|joins|Becoming_a_member) (e10|realizes|Know) (e11|raids|Attack) (e12|recruiting|Becoming_a_member) (e13|support|Supporting) (e14|attack|Attack) (e15|imprisoned|Arrest) (e16|take|Conquering) (e17|out|A
…(truncated, full length 1489 chars)
```

**temporal_causal_joint**
```
EVENTS:
(e1|ruled|Control) (e2|arrested|Arrest) (e3|escape|Escaping) (e4|negotiate|Communication) (e5|hire|Employment) (e6|re|Cause_to_amalgamate) (e7|leaving|Departing) (e8|named|Name_conferral) (e9|joins|Becoming_a_member) (e10|realizes|Know) (e11|raids|Attack) (e12|recruiting|Becoming_a_member) (e13|support|Supporting) (e14|attack|Attack) (e15|imprisoned|Arrest) (e16|take|Conquering) (e17|out|A
…(truncated, full length 1203 chars)
```

### 1000826 / en (en) — events: 35, relations: {'temporal': 15, 'causal': 8, 'joint': 6, 'indep(t)': 15, 'indep(c)': 8}

**events_only**
```
EVENTS:
(e1|capture|Conquering) (e2|revolutionary|Change_of_leadership) (e3|opposing|Agree_or_refuse_to_act) (e4|going|Self_motion) (e5|to|Motion) (e6|gives|Giving) (e7|demands|Request) (e8|used|Using) (e9|crosses|Motion_directional) (e10|told|Telling) (e11|finds|Know) (e12|using|Using) (e13|agrees|Agree_or_refuse_to_act) (e14|rescue|Rescuing) (e15|uses|Using) (e16|saved|Rescuing) (e17|called|Name
…(truncated, full length 890 chars)
```

**temporal**
```
EVENTS:
(e1|capture|Conquering) (e2|revolutionary|Change_of_leadership) (e3|opposing|Agree_or_refuse_to_act) (e4|going|Self_motion) (e5|to|Motion) (e6|gives|Giving) (e7|demands|Request) (e8|used|Using) (e9|crosses|Motion_directional) (e10|told|Telling) (e11|finds|Know) (e12|using|Using) (e13|agrees|Agree_or_refuse_to_act) (e14|rescue|Rescuing) (e15|uses|Using) (e16|saved|Rescuing) (e17|called|Name
…(truncated, full length 1182 chars)
```

**causal**
```
EVENTS:
(e1|capture|Conquering) (e2|revolutionary|Change_of_leadership) (e3|opposing|Agree_or_refuse_to_act) (e4|going|Self_motion) (e5|to|Motion) (e6|gives|Giving) (e7|demands|Request) (e8|used|Using) (e9|crosses|Motion_directional) (e10|told|Telling) (e11|finds|Know) (e12|using|Using) (e13|agrees|Agree_or_refuse_to_act) (e14|rescue|Rescuing) (e15|uses|Using) (e16|saved|Rescuing) (e17|called|Name
…(truncated, full length 1047 chars)
```

**temporal_causal_independent**
```
EVENTS:
(e1|capture|Conquering) (e2|revolutionary|Change_of_leadership) (e3|opposing|Agree_or_refuse_to_act) (e4|going|Self_motion) (e5|to|Motion) (e6|gives|Giving) (e7|demands|Request) (e8|used|Using) (e9|crosses|Motion_directional) (e10|told|Telling) (e11|finds|Know) (e12|using|Using) (e13|agrees|Agree_or_refuse_to_act) (e14|rescue|Rescuing) (e15|uses|Using) (e16|saved|Rescuing) (e17|called|Name
…(truncated, full length 1339 chars)
```

**temporal_causal_joint**
```
EVENTS:
(e1|capture|Conquering) (e2|revolutionary|Change_of_leadership) (e3|opposing|Agree_or_refuse_to_act) (e4|going|Self_motion) (e5|to|Motion) (e6|gives|Giving) (e7|demands|Request) (e8|used|Using) (e9|crosses|Motion_directional) (e10|told|Telling) (e11|finds|Know) (e12|using|Using) (e13|agrees|Agree_or_refuse_to_act) (e14|rescue|Rescuing) (e15|uses|Using) (e16|saved|Rescuing) (e17|called|Name
…(truncated, full length 1060 chars)
```

### 1000826 / it (it) — events: 34, relations: {'temporal': 29, 'causal': 23, 'joint': 32, 'indep(t)': 29, 'indep(c)': 23}

**events_only**
```
EVENTS:
(e1|capture|Conquering) (e2|revolutionary|Change_of_leadership) (e3|unite|Cause_to_amalgamate) (e4|opposed|Agree_or_refuse_to_act) (e5|arrested|Arrest) (e6|hands|Giving) (e7|asks|Request) (e8|buy|Commerce_buy) (e9|decides|Deciding) (e10|cross|Motion_directional) (e11|saves|Rescuing) (e12|manipulating|Control) (e13|approve|Agree_or_refuse_to_act) (e14|uses|Using) (e15|rescued|Rescuing) (e16
…(truncated, full length 842 chars)
```

**temporal**
```
EVENTS:
(e1|capture|Conquering) (e2|revolutionary|Change_of_leadership) (e3|unite|Cause_to_amalgamate) (e4|opposed|Agree_or_refuse_to_act) (e5|arrested|Arrest) (e6|hands|Giving) (e7|asks|Request) (e8|buy|Commerce_buy) (e9|decides|Deciding) (e10|cross|Motion_directional) (e11|saves|Rescuing) (e12|manipulating|Control) (e13|approve|Agree_or_refuse_to_act) (e14|uses|Using) (e15|rescued|Rescuing) (e16
…(truncated, full length 1394 chars)
```

**causal**
```
EVENTS:
(e1|capture|Conquering) (e2|revolutionary|Change_of_leadership) (e3|unite|Cause_to_amalgamate) (e4|opposed|Agree_or_refuse_to_act) (e5|arrested|Arrest) (e6|hands|Giving) (e7|asks|Request) (e8|buy|Commerce_buy) (e9|decides|Deciding) (e10|cross|Motion_directional) (e11|saves|Rescuing) (e12|manipulating|Control) (e13|approve|Agree_or_refuse_to_act) (e14|uses|Using) (e15|rescued|Rescuing) (e16
…(truncated, full length 1262 chars)
```

**temporal_causal_independent**
```
EVENTS:
(e1|capture|Conquering) (e2|revolutionary|Change_of_leadership) (e3|unite|Cause_to_amalgamate) (e4|opposed|Agree_or_refuse_to_act) (e5|arrested|Arrest) (e6|hands|Giving) (e7|asks|Request) (e8|buy|Commerce_buy) (e9|decides|Deciding) (e10|cross|Motion_directional) (e11|saves|Rescuing) (e12|manipulating|Control) (e13|approve|Agree_or_refuse_to_act) (e14|uses|Using) (e15|rescued|Rescuing) (e16
…(truncated, full length 1814 chars)
```

**temporal_causal_joint**
```
EVENTS:
(e1|capture|Conquering) (e2|revolutionary|Change_of_leadership) (e3|unite|Cause_to_amalgamate) (e4|opposed|Agree_or_refuse_to_act) (e5|arrested|Arrest) (e6|hands|Giving) (e7|asks|Request) (e8|buy|Commerce_buy) (e9|decides|Deciding) (e10|cross|Motion_directional) (e11|saves|Rescuing) (e12|manipulating|Control) (e13|approve|Agree_or_refuse_to_act) (e14|uses|Using) (e15|rescued|Rescuing) (e16
…(truncated, full length 1642 chars)
```

### 1001400 / en (en) — events: 42, relations: {'temporal': 40, 'causal': 7, 'joint': 40, 'indep(t)': 40, 'indep(c)': 7}

**events_only**
```
EVENTS:
(e1|starts|Process_start) (e2|rec|Statement) (e3|ount|Statement) (e4|massacre|Killing) (e5|says|Statement) (e6|maintained|Preserving) (e7|contact|Communication) (e8|mentioning|Statement) (e9|maintains|Statement) (e10|gave rise to|Causation) (e11|acquired|Getting) (e12|leaving|Departing) (e13|dedicated|Giving) (e14|mentions|Statement) (e15|rejected|Agree_or_refuse_to_act) (e16|election|Choo
…(truncated, full length 1146 chars)
```

**temporal**
```
EVENTS:
(e1|starts|Process_start) (e2|rec|Statement) (e3|ount|Statement) (e4|massacre|Killing) (e5|says|Statement) (e6|maintained|Preserving) (e7|contact|Communication) (e8|mentioning|Statement) (e9|maintains|Statement) (e10|gave rise to|Causation) (e11|acquired|Getting) (e12|leaving|Departing) (e13|dedicated|Giving) (e14|mentions|Statement) (e15|rejected|Agree_or_refuse_to_act) (e16|election|Choo
…(truncated, full length 1907 chars)
```

**causal**
```
EVENTS:
(e1|starts|Process_start) (e2|rec|Statement) (e3|ount|Statement) (e4|massacre|Killing) (e5|says|Statement) (e6|maintained|Preserving) (e7|contact|Communication) (e8|mentioning|Statement) (e9|maintains|Statement) (e10|gave rise to|Causation) (e11|acquired|Getting) (e12|leaving|Departing) (e13|dedicated|Giving) (e14|mentions|Statement) (e15|rejected|Agree_or_refuse_to_act) (e16|election|Choo
…(truncated, full length 1287 chars)
```

**temporal_causal_independent**
```
EVENTS:
(e1|starts|Process_start) (e2|rec|Statement) (e3|ount|Statement) (e4|massacre|Killing) (e5|says|Statement) (e6|maintained|Preserving) (e7|contact|Communication) (e8|mentioning|Statement) (e9|maintains|Statement) (e10|gave rise to|Causation) (e11|acquired|Getting) (e12|leaving|Departing) (e13|dedicated|Giving) (e14|mentions|Statement) (e15|rejected|Agree_or_refuse_to_act) (e16|election|Choo
…(truncated, full length 2048 chars)
```

**temporal_causal_joint**
```
EVENTS:
(e1|starts|Process_start) (e2|rec|Statement) (e3|ount|Statement) (e4|massacre|Killing) (e5|says|Statement) (e6|maintained|Preserving) (e7|contact|Communication) (e8|mentioning|Statement) (e9|maintains|Statement) (e10|gave rise to|Causation) (e11|acquired|Getting) (e12|leaving|Departing) (e13|dedicated|Giving) (e14|mentions|Statement) (e15|rejected|Agree_or_refuse_to_act) (e16|election|Choo
…(truncated, full length 2140 chars)
```

### 1001400 / fr (fr) — events: 56, relations: {'temporal': 55, 'causal': 19, 'joint': 28, 'indep(t)': 55, 'indep(c)': 19}

**events_only**
```
EVENTS:
(e1|begins|Process_start) (e2|incorporation|Cause_to_amalgamate) (e3|mentions|Statement) (e4|maintained|Preserving) (e5|played|Participation) (e6|covers|Hiding_objects) (e7|collapse|Destroying) (e8|argues|Quarreling) (e9|given|Giving) (e10|rise to|Causation) (e11|acquired|Getting) (e12|leaving|Departing) (e13|provide|Supply) (e14|support|Supporting) (e15|devoted|Giving) (e16|tells|Telling)
…(truncated, full length 1611 chars)
```

**temporal**
```
EVENTS:
(e1|begins|Process_start) (e2|incorporation|Cause_to_amalgamate) (e3|mentions|Statement) (e4|maintained|Preserving) (e5|played|Participation) (e6|covers|Hiding_objects) (e7|collapse|Destroying) (e8|argues|Quarreling) (e9|given|Giving) (e10|rise to|Causation) (e11|acquired|Getting) (e12|leaving|Departing) (e13|provide|Supply) (e14|support|Supporting) (e15|devoted|Giving) (e16|tells|Telling)
…(truncated, full length 2651 chars)
```

**causal**
```
EVENTS:
(e1|begins|Process_start) (e2|incorporation|Cause_to_amalgamate) (e3|mentions|Statement) (e4|maintained|Preserving) (e5|played|Participation) (e6|covers|Hiding_objects) (e7|collapse|Destroying) (e8|argues|Quarreling) (e9|given|Giving) (e10|rise to|Causation) (e11|acquired|Getting) (e12|leaving|Departing) (e13|provide|Supply) (e14|support|Supporting) (e15|devoted|Giving) (e16|tells|Telling)
…(truncated, full length 1966 chars)
```

**temporal_causal_independent**
```
EVENTS:
(e1|begins|Process_start) (e2|incorporation|Cause_to_amalgamate) (e3|mentions|Statement) (e4|maintained|Preserving) (e5|played|Participation) (e6|covers|Hiding_objects) (e7|collapse|Destroying) (e8|argues|Quarreling) (e9|given|Giving) (e10|rise to|Causation) (e11|acquired|Getting) (e12|leaving|Departing) (e13|provide|Supply) (e14|support|Supporting) (e15|devoted|Giving) (e16|tells|Telling)
…(truncated, full length 3006 chars)
```

**temporal_causal_joint**
```
EVENTS:
(e1|begins|Process_start) (e2|incorporation|Cause_to_amalgamate) (e3|mentions|Statement) (e4|maintained|Preserving) (e5|played|Participation) (e6|covers|Hiding_objects) (e7|collapse|Destroying) (e8|argues|Quarreling) (e9|given|Giving) (e10|rise to|Causation) (e11|acquired|Getting) (e12|leaving|Departing) (e13|provide|Supply) (e14|support|Supporting) (e15|devoted|Giving) (e16|tells|Telling)
…(truncated, full length 2505 chars)
```

### 100156260 / en (en) — events: 96, relations: {'temporal': 42, 'causal': 28, 'joint': 36, 'indep(t)': 42, 'indep(c)': 28}

**events_only**
```
EVENTS:
(e1|running|Self_motion) (e2|perform|Hold) (e3|produces|Creating) (e4|electro|Bodily_harm) (e5|cute|Bodily_harm) (e6|control|Control) (e7|suffering|Bodily_harm) (e8|covered|Hiding_objects) (e9|dation|Bodily_harm) (e10|manipulate|Control) (e11|conflict|Hostile_encounter) (e12|puts|Causation) (e13|proposes|Statement) (e14|travel|Traveling) (e15|proposes|Statement) (e16|find|Know) (e17|on|Pla
…(truncated, full length 2260 chars)
```

**temporal**
```
EVENTS:
(e1|running|Self_motion) (e2|perform|Hold) (e3|produces|Creating) (e4|electro|Bodily_harm) (e5|cute|Bodily_harm) (e6|control|Control) (e7|suffering|Bodily_harm) (e8|covered|Hiding_objects) (e9|dation|Bodily_harm) (e10|manipulate|Control) (e11|conflict|Hostile_encounter) (e12|puts|Causation) (e13|proposes|Statement) (e14|travel|Traveling) (e15|proposes|Statement) (e16|find|Know) (e17|on|Pla
…(truncated, full length 3078 chars)
```

**causal**
```
EVENTS:
(e1|running|Self_motion) (e2|perform|Hold) (e3|produces|Creating) (e4|electro|Bodily_harm) (e5|cute|Bodily_harm) (e6|control|Control) (e7|suffering|Bodily_harm) (e8|covered|Hiding_objects) (e9|dation|Bodily_harm) (e10|manipulate|Control) (e11|conflict|Hostile_encounter) (e12|puts|Causation) (e13|proposes|Statement) (e14|travel|Traveling) (e15|proposes|Statement) (e16|find|Know) (e17|on|Pla
…(truncated, full length 2772 chars)
```

**temporal_causal_independent**
```
EVENTS:
(e1|running|Self_motion) (e2|perform|Hold) (e3|produces|Creating) (e4|electro|Bodily_harm) (e5|cute|Bodily_harm) (e6|control|Control) (e7|suffering|Bodily_harm) (e8|covered|Hiding_objects) (e9|dation|Bodily_harm) (e10|manipulate|Control) (e11|conflict|Hostile_encounter) (e12|puts|Causation) (e13|proposes|Statement) (e14|travel|Traveling) (e15|proposes|Statement) (e16|find|Know) (e17|on|Pla
…(truncated, full length 3590 chars)
```

**temporal_causal_joint**
```
EVENTS:
(e1|running|Self_motion) (e2|perform|Hold) (e3|produces|Creating) (e4|electro|Bodily_harm) (e5|cute|Bodily_harm) (e6|control|Control) (e7|suffering|Bodily_harm) (e8|covered|Hiding_objects) (e9|dation|Bodily_harm) (e10|manipulate|Control) (e11|conflict|Hostile_encounter) (e12|puts|Causation) (e13|proposes|Statement) (e14|travel|Traveling) (e15|proposes|Statement) (e16|find|Know) (e17|on|Pla
…(truncated, full length 3317 chars)
```

### 100156260 / fr (fr) — events: 20, relations: {'temporal': 7, 'causal': 6, 'joint': 6, 'indep(t)': 7, 'indep(c)': 6}

**events_only**
```
EVENTS:
(e1|produces|Creating) (e2|control|Control) (e3|suffers|Bodily_harm) (e4|endowed|Supply) (e5|move|Motion) (e6|conflict|Hostile_encounter) (e7|beginning|Process_start) (e8|put an end|Process_end) (e9|to|Causation) (e10|made|Causation) (e11|proposes|Statement) (e12|go to|Motion) (e13|suggests|Convincing) (e14|finding|Know) (e15|put on|Placing) (e16|predicted|Statement) (e17|suicide|Killing) 
…(truncated, full length 474 chars)
```

**temporal**
```
EVENTS:
(e1|produces|Creating) (e2|control|Control) (e3|suffers|Bodily_harm) (e4|endowed|Supply) (e5|move|Motion) (e6|conflict|Hostile_encounter) (e7|beginning|Process_start) (e8|put an end|Process_end) (e9|to|Causation) (e10|made|Causation) (e11|proposes|Statement) (e12|go to|Motion) (e13|suggests|Convincing) (e14|finding|Know) (e15|put on|Placing) (e16|predicted|Statement) (e17|suicide|Killing) 
…(truncated, full length 614 chars)
```

**causal**
```
EVENTS:
(e1|produces|Creating) (e2|control|Control) (e3|suffers|Bodily_harm) (e4|endowed|Supply) (e5|move|Motion) (e6|conflict|Hostile_encounter) (e7|beginning|Process_start) (e8|put an end|Process_end) (e9|to|Causation) (e10|made|Causation) (e11|proposes|Statement) (e12|go to|Motion) (e13|suggests|Convincing) (e14|finding|Know) (e15|put on|Placing) (e16|predicted|Statement) (e17|suicide|Killing) 
…(truncated, full length 596 chars)
```

**temporal_causal_independent**
```
EVENTS:
(e1|produces|Creating) (e2|control|Control) (e3|suffers|Bodily_harm) (e4|endowed|Supply) (e5|move|Motion) (e6|conflict|Hostile_encounter) (e7|beginning|Process_start) (e8|put an end|Process_end) (e9|to|Causation) (e10|made|Causation) (e11|proposes|Statement) (e12|go to|Motion) (e13|suggests|Convincing) (e14|finding|Know) (e15|put on|Placing) (e16|predicted|Statement) (e17|suicide|Killing) 
…(truncated, full length 736 chars)
```

**temporal_causal_joint**
```
EVENTS:
(e1|produces|Creating) (e2|control|Control) (e3|suffers|Bodily_harm) (e4|endowed|Supply) (e5|move|Motion) (e6|conflict|Hostile_encounter) (e7|beginning|Process_start) (e8|put an end|Process_end) (e9|to|Causation) (e10|made|Causation) (e11|proposes|Statement) (e12|go to|Motion) (e13|suggests|Convincing) (e14|finding|Know) (e15|put on|Placing) (e16|predicted|Statement) (e17|suicide|Killing) 
…(truncated, full length 658 chars)
```

### 100156260 / it (it) — events: 73, relations: {'temporal': 67, 'causal': 56, 'joint': 59, 'indep(t)': 67, 'indep(c)': 56}

**events_only**
```
EVENTS:
(e1|take place|Process_start) (e2|perform|Hold) (e3|produces|Creating) (e4|electro|Bodily_harm) (e5|cuting|Bodily_harm) (e6|controlling|Control) (e7|affected|Influence) (e8|endowed|Supply) (e9|control|Control) (e10|conflict|Hostile_encounter) (e11|threatens|Warning) (e12|proposes|Statement) (e13|collecting|Come_together) (e14|Searching|Scrutiny) (e15|escaping|Escaping) (e16|decide|Deciding
…(truncated, full length 1743 chars)
```

**temporal**
```
EVENTS:
(e1|take place|Process_start) (e2|perform|Hold) (e3|produces|Creating) (e4|electro|Bodily_harm) (e5|cuting|Bodily_harm) (e6|controlling|Control) (e7|affected|Influence) (e8|endowed|Supply) (e9|control|Control) (e10|conflict|Hostile_encounter) (e11|threatens|Warning) (e12|proposes|Statement) (e13|collecting|Come_together) (e14|Searching|Scrutiny) (e15|escaping|Escaping) (e16|decide|Deciding
…(truncated, full length 3022 chars)
```

**causal**
```
EVENTS:
(e1|take place|Process_start) (e2|perform|Hold) (e3|produces|Creating) (e4|electro|Bodily_harm) (e5|cuting|Bodily_harm) (e6|controlling|Control) (e7|affected|Influence) (e8|endowed|Supply) (e9|control|Control) (e10|conflict|Hostile_encounter) (e11|threatens|Warning) (e12|proposes|Statement) (e13|collecting|Come_together) (e14|Searching|Scrutiny) (e15|escaping|Escaping) (e16|decide|Deciding
…(truncated, full length 2766 chars)
```

**temporal_causal_independent**
```
EVENTS:
(e1|take place|Process_start) (e2|perform|Hold) (e3|produces|Creating) (e4|electro|Bodily_harm) (e5|cuting|Bodily_harm) (e6|controlling|Control) (e7|affected|Influence) (e8|endowed|Supply) (e9|control|Control) (e10|conflict|Hostile_encounter) (e11|threatens|Warning) (e12|proposes|Statement) (e13|collecting|Come_together) (e14|Searching|Scrutiny) (e15|escaping|Escaping) (e16|decide|Deciding
…(truncated, full length 4045 chars)
```

**temporal_causal_joint**
```
EVENTS:
(e1|take place|Process_start) (e2|perform|Hold) (e3|produces|Creating) (e4|electro|Bodily_harm) (e5|cuting|Bodily_harm) (e6|controlling|Control) (e7|affected|Influence) (e8|endowed|Supply) (e9|control|Control) (e10|conflict|Hostile_encounter) (e11|threatens|Warning) (e12|proposes|Statement) (e13|collecting|Come_together) (e14|Searching|Scrutiny) (e15|escaping|Escaping) (e16|decide|Deciding
…(truncated, full length 3265 chars)
```

In [10]:
pass_count, fail_count, fails = 0, 0, []
for _, row in df.iterrows():
    events_only = row["linearized_events_only"]
    checks = {
        "temporal_has_events_prefix":  row["conditions"]["temporal"]                  ["linearized"].startswith(events_only),
        "causal_has_events_prefix":    row["conditions"]["causal"]                    ["linearized"].startswith(events_only),
        "indep_has_events_prefix":     row["conditions"]["temporal_causal_independent"]["linearized"].startswith(events_only),
        "joint_has_events_prefix":     row["conditions"]["temporal_causal_joint"]     ["linearized"].startswith(events_only),
        "indep_has_temporal_header":   "TEMPORAL:"      in row["conditions"]["temporal_causal_independent"]["linearized"],
        "indep_has_causal_header":     "CAUSAL:"        in row["conditions"]["temporal_causal_independent"]["linearized"],
        "joint_has_temperocausal":     "TEMPEROCAUSAL:" in row["conditions"]["temporal_causal_joint"]      ["linearized"],
    }
    if all(checks.values()):
        pass_count += 1
    else:
        fail_count += 1
        fails.append((row["wikidata_id"], row["summary_id"], [k for k, v in checks.items() if not v]))

print(f"PASS: {pass_count}    FAIL: {fail_count}")
for wid, sid, broken in fails:
    print(f"  {wid}/{sid}: {broken}")


PASS: 10    FAIL: 0


## 8. UNI-14 — Embeddings spot-check

Validate the embeddings produced by `models/embed/encode_embeddings.py` + merged by `src/build_embeddings.py`. Three checks: presence per encoder, the unit-norm invariant (every L2-normalized vector must have norm ≈ 1), and a failure-mode rows audit (cross-references `conditions.<c>.source` and `parse_error`; per UNI-82 these will be filtered at retrieval, not here).

In [7]:
# Load + presence per encoder.
import json
from pathlib import Path

rows = [json.loads(l) for l in Path("../data/results/experiment.jsonl").read_text().splitlines() if l]
print(f"Loaded {len(rows)} rows.")

encoder_keys = sorted({k for r in rows for k in r.get("embeddings", {}).keys()})
print(f"Encoders present: {encoder_keys}")

for enc in encoder_keys:
    missing = [(r["wikidata_id"], r["summary_id"]) for r in rows if enc not in r.get("embeddings", {})]
    print(f"  [{enc}] missing on {len(missing)} rows: {missing[:5]}{'…' if len(missing) > 5 else ''}")


Loaded 10 rows.
Encoders present: ['e5_mistral', 'sbert_mpnet']
  [e5_mistral] missing on 0 rows: []
  [sbert_mpnet] missing on 0 rows: []


In [8]:
# Norm invariant + degeneracy + failure-mode audit.
import json
from pathlib import Path

import numpy as np

rows = [json.loads(l) for l in Path("../data/results/experiment.jsonl").read_text().splitlines() if l]
encoder_keys = sorted({k for r in rows for k in r.get("embeddings", {}).keys()})
conditions = (
    "raw_text", "events_only", "temporal", "causal",
    "temporal_causal_independent", "temporal_causal_joint",
)

hard_failures = 0
for enc in encoder_keys:
    for cond in conditions:
        bad = []
        for r in rows:
            if enc not in r.get("embeddings", {}):
                continue
            v = np.asarray(r["embeddings"][enc]["vectors"][cond])
            n = float(np.linalg.norm(v))
            if not (1 - 1e-3 <= n <= 1 + 1e-3):
                bad.append((r["wikidata_id"], r["summary_id"], n))
        if bad:
            hard_failures += len(bad)
            print(f"[{enc}/{cond}] {len(bad)} norm violations: {bad[:3]}…")
print(f"\nHard invariant failures (norm not ≈ 1): {hard_failures}  ({'PASS' if hard_failures == 0 else 'FAIL'})")

print("\nFailure-mode (row, condition) triples — embeddings populated, retrieval will skip (UNI-82):")
for r in rows:
    for cond in ("temporal", "causal", "temporal_causal_independent", "temporal_causal_joint"):
        block = r["conditions"][cond]
        if block.get("source") not in {"llama", "composed"} or block.get("parse_error"):
            has_emb = bool(encoder_keys) and all(
                cond in r.get("embeddings", {}).get(enc, {}).get("vectors", {})
                for enc in encoder_keys
            )
            print(f"  {r['wikidata_id']}/{r['summary_id']}/{cond}  "
                  f"source={block.get('source')!r}  parse_error={block.get('parse_error')!r}  "
                  f"embedded={has_emb}")



Hard invariant failures (norm not ≈ 1): 0  (PASS)

Failure-mode (row, condition) triples — embeddings populated, retrieval will skip (UNI-82):
  1000352/en/temporal_causal_joint  source='llama'  parse_error='JSONDecodeError: Expecting value: line 1 column 1 (char 0)'  embedded=True


## 9. UNI-17 — Baselines spot-check

Validate the BoW and TF-IDF vectors produced by `notebooks/pipeline.ipynb` §7. Four checks: presence per method, the unit-norm invariant (sklearn L2 in double precision — tight `1e-6` tolerance, unlike the bf16 embedding case in §8), sparsity sanity (no all-zero vectors), and top-token inspection via the vocab side-file.

In [1]:
# Presence, L2-norm invariant, sparsity, top-token inspection for baselines.

import json
from pathlib import Path

import numpy as np

EXP_PATH   = Path("../data/results/experiment.jsonl")
VOCAB_PATH = Path("../data/intermediate/baselines/vocab.json")

rows  = [json.loads(l) for l in EXP_PATH.read_text(encoding="utf-8").splitlines() if l]
vocab = json.loads(VOCAB_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(rows)} rows.")

# 1. Presence per method.
for method in ("bow", "tfidf"):
    missing = [(r["wikidata_id"], r["summary_id"])
               for r in rows
               if method not in r.get("baselines", {})
               or "vector" not in r["baselines"][method]]
    print(f"  [{method}] missing on {len(missing)} rows: {missing[:5]}{'\u2026' if len(missing) > 5 else ''}")

# 2. L2-norm invariant + dim consistency + degenerate (all-zero) check.
TOL = 1e-6
for method in ("bow", "tfidf"):
    dims, bad_norms, zero_vecs = set(), [], []
    for r in rows:
        b = r["baselines"][method]
        dims.add(b["dim"])
        vals = np.asarray(b["vector"]["values"], dtype=np.float64)
        if vals.size == 0:
            zero_vecs.append((r["wikidata_id"], r["summary_id"]))
            continue
        n = float(np.sqrt(np.sum(vals ** 2)))
        if not (1 - TOL <= n <= 1 + TOL):
            bad_norms.append((r["wikidata_id"], r["summary_id"], n))
    print(f"\n[{method}] dim set = {dims}  (expect singleton)")
    print(f"  L2-norm violations (|\u2016v\u2016-1| > {TOL}): {len(bad_norms)}  ({'PASS' if not bad_norms else 'FAIL'})")
    if bad_norms:
        print(f"    examples: {bad_norms[:3]}")
    print(f"  Degenerate (all-zero) vectors:        {len(zero_vecs)}  ({'PASS' if not zero_vecs else 'FAIL'})")
    if zero_vecs:
        print(f"    examples: {zero_vecs[:3]}")

# 3. Sparsity stats per method.
print()
for method in ("bow", "tfidf"):
    nnz = np.array([len(r["baselines"][method]["vector"]["indices"]) for r in rows])
    dim = next(iter({r["baselines"][method]["dim"] for r in rows}))
    print(f"[{method}] nnz mean={nnz.mean():.1f}  p50={int(np.median(nnz))}  "
          f"p95={int(np.quantile(nnz, 0.95))}  max={nnz.max()}   "
          f"density mean={nnz.mean()/dim:.4%}")

# 4. Top-10 token inspection for 3 sample rows (deterministic \u2014 file order).
from IPython.display import display, Markdown
TOP_K = 10
for r in rows[:3]:
    md = [f"### {r['wikidata_id']} / {r['summary_id']} ({r['lang']})"]
    for method in ("bow", "tfidf"):
        b = r["baselines"][method]
        pairs = sorted(zip(b["vector"]["indices"], b["vector"]["values"]),
                       key=lambda iv: -iv[1])[:TOP_K]
        rendered = ", ".join(f"`{vocab[method][i]}`={v:.4f}" for i, v in pairs)
        md.append(f"**top-{TOP_K} {method}**: {rendered}")
    display(Markdown("\n\n".join(md)))


Loaded 10 rows.
  [bow] missing on 0 rows: []
  [tfidf] missing on 0 rows: []

[bow] dim set = {1267}  (expect singleton)
  L2-norm violations (|‖v‖-1| > 1e-06): 0  (PASS)
  Degenerate (all-zero) vectors:        0  (PASS)

[tfidf] dim set = {1267}  (expect singleton)
  L2-norm violations (|‖v‖-1| > 1e-06): 0  (PASS)
  Degenerate (all-zero) vectors:        0  (PASS)

[bow] nnz mean=208.5  p50=234  p95=356  max=388   density mean=16.4562%
[tfidf] nnz mean=208.5  p50=234  p95=356  max=388   density mean=16.4562%


### 1000352 / de (de)

**top-10 bow**: `the`=0.6729, `to`=0.3106, `and`=0.2847, `him`=0.1941, `fairy`=0.1812, `he`=0.1682, `of`=0.1682, `will`=0.1682, `jacob`=0.1424, `his`=0.1294

**top-10 tfidf**: `the`=0.4883, `fairy`=0.3023, `jacob`=0.2375, `and`=0.2263, `to`=0.2254, `will`=0.2183, `him`=0.2045, `dark`=0.1943, `goyl`=0.1511, `he`=0.1337

### 1000352 / en (en)

**top-10 bow**: `the`=0.7301, `to`=0.3245, `jacob`=0.2559, `will`=0.2434, `and`=0.2309, `he`=0.1186, `is`=0.1186, `of`=0.1186, `that`=0.0998, `clara`=0.0936

**top-10 tfidf**: `the`=0.5388, `jacob`=0.4342, `will`=0.3212, `to`=0.2395, `and`=0.1866, `goyl`=0.1588, `clara`=0.1588, `fairy`=0.1483, `dark`=0.1059, `valiant`=0.1019

### 1000826 / de (de)

**top-10 bow**: `of`=0.4201, `the`=0.4201, `chris`=0.2801, `been`=0.1400, `by`=0.1400, `dictator`=0.1400, `díaz`=0.1400, `enlists`=0.1400, `goes`=0.1400, `has`=0.1400

**top-10 tfidf**: `chris`=0.2588, `of`=0.2417, `the`=0.2417, `taken`=0.2179, `troops`=0.2179, `dictator`=0.2179, `porfirio`=0.2179, `enlists`=0.2179, `more`=0.2179, `mercenaries`=0.2179